In [16]:
# Zelle 1: Imports
import json
from pathlib import Path
import re
from collections import Counter

# Zelle 2: Initialisierung der Ergebnislisten
auftraege_counts = []
bestellpositionen_counts = []

# Zelle 3: Alle JSON-Dateien im aktuellen Ordner durchsuchen
data_folder = Path(".")  # optional: Pfad anpassen, z. B. Path("data")
json_files = list(data_folder.glob("*.json"))

# Sortieren der Dateien nach der Zahl (zweistellig) hinter "KW"

def extract_kw_number(filename):
    match = re.search(r"KW(\d+)", filename)
    return int(match.group(1)) if match else -1

def extract_prefix_number(filename):
    # Extrahiere die Zahl vor _KW, z.B. 7 in Construction_RealLife_2024_7_KW29.json
    match = re.search(r"_(\d+)_KW", filename)
    return int(match.group(1)) if match else -1

# Finde alle KWs, die mehrfach vorkommen

kw_list = [extract_kw_number(p.name) for p in json_files]
kw_counts = Counter(kw_list)
doppelte_kws = [kw for kw, count in kw_counts.items() if count > 1]

if doppelte_kws:
    print("Doppelt vorkommende KWs:", doppelte_kws)
else:
    print("Keine doppelten KWs gefunden.")

json_files.sort(key=lambda p: (extract_kw_number(p.name), extract_prefix_number(p.name)))

json_files.sort(key=lambda p: extract_kw_number(p.name))


# Zelle 4: Durch alle Dateien iterieren
for filepath in json_files:
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        auftraege = data.get("Auftraege", [])
        bestellpositionen = data.get("Bestellpositionen", [])
        workers = data.get("Arbeiter", [])
        machines = data.get("Maschinen", [])
        attachments = data.get("Anbaugeraete", [])

        num_auftraege = len(auftraege)
        num_bestellpositionen = len(bestellpositionen)
        num_workers = len(workers)
        num_machines = len(machines)
        num_attachments = len(attachments)

        auftraege_counts.append(num_auftraege)
        bestellpositionen_counts.append(num_bestellpositionen)

        print(f"{filepath.name}:")
        print(f"  Anzahl der Aufträge: {num_auftraege}")
        print(f"  Anzahl der Bestellpositionen: {num_bestellpositionen}")
        print(f"  Anzahl der Maschinen: {num_machines}")
        print(f"  Anzahl der Arbeiter: {num_workers}")
        print(f"  Anzahl der Anbaugeräte: {num_attachments}")
        print("-" * 40)

    except FileNotFoundError:
        print(f"Datei {filepath.name} nicht gefunden.")
    except json.JSONDecodeError:
        print(f"Datei {filepath.name} konnte nicht als JSON geladen werden.")

# Zelle 5: Min/Max-Werte ausgeben
if auftraege_counts and bestellpositionen_counts:
    print("Gesamtübersicht:")
    print(f"  Aufträge: min = {min(auftraege_counts)}, max = {max(auftraege_counts)}")
    print(f"  Bestellpositionen: min = {min(bestellpositionen_counts)}, max = {max(bestellpositionen_counts)}")

Doppelt vorkommende KWs: [35, 31, 40]
Construction_RealLife_2024_7_KW27.json:
  Anzahl der Aufträge: 82
  Anzahl der Bestellpositionen: 466
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 391
----------------------------------------
Construction_RealLife_2024_7_KW28.json:
  Anzahl der Aufträge: 81
  Anzahl der Bestellpositionen: 460
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 391
----------------------------------------
Construction_RealLife_2024_7_KW29.json:
  Anzahl der Aufträge: 133
  Anzahl der Bestellpositionen: 875
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 391
----------------------------------------
Construction_RealLife_2024_7_KW30.json:
  Anzahl der Aufträge: 125
  Anzahl der Bestellpositionen: 817
  Anzahl der Maschinen: 65
  Anzahl der Arbeiter: 118
  Anzahl der Anbaugeräte: 391
----------------------------------------
Construction_RealLife_2024_7_KW31.json:
  Anzahl der

In [17]:
import json

# Lade die JSON-Datei
with open("Construction_RealLife_2024_10_1.json", "r") as f:
    data = json.load(f)

# Extrahiere alle Baustellenstandorte
baustellen_standorte = [
    (item["Standort"]["Item1"], item["Standort"]["Item2"])
    for item in data["Auftraege"]
]

# Extrahiere alle Wohnorte der Arbeiter
arbeiter_wohnorte = [
    (worker["Wohnort"]["Item1"], worker["Wohnort"]["Item2"])
    for worker in data["Arbeiter"]
]

import folium

# Mittelpunkt berechnen (z. B. Mittelwert aller Koordinaten)
all_coords = baustellen_standorte + arbeiter_wohnorte
avg_lat = sum(lat for lat, _ in all_coords) / len(all_coords)
avg_lon = sum(lon for _, lon in all_coords) / len(all_coords)

# Erstelle die Karte zentriert auf den Mittelwert
karte = folium.Map(location=[avg_lat, avg_lon], zoom_start=8)

# Füge Baustellen als blaue Marker hinzu
for lat, lon in baustellen_standorte:
    folium.Marker(
        location=[lat, lon],
        popup="Baustelle",
        icon=folium.Icon(color="blue", icon="wrench", prefix="fa")
    ).add_to(karte)

# Füge Arbeiter als grüne Marker hinzu
for lat, lon in arbeiter_wohnorte:
    folium.Marker(
        location=[lat, lon],
        popup="Arbeiter",
        icon=folium.Icon(color="green", icon="user", prefix="fa")
    ).add_to(karte)

# Karte anzeigen
karte

FileNotFoundError: [Errno 2] No such file or directory: 'Construction_RealLife_2024_10_1.json'